<a href="https://colab.research.google.com/github/LostRuins/koboldcpp/blob/concedo/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Welcome to the Official KoboldCpp Colab Notebook
It's really easy to get started. Just press the two **Play** buttons below, and then connect to the **Cloudflare URL** shown at the end.
You can select a model from the dropdown, or enter a **custom URL** to a GGUF model (Example: `https://huggingface.co/KoboldAI/LLaMA2-13B-Tiefighter-GGUF/resolve/main/LLaMA2-13B-Tiefighter.Q4_K_M.gguf`)

Named models in the Model selector automatically use the optimal context settings as long as you do not load additional models such as an image generation model.

**Keep this page open and occationally check for captcha's so that your AI is not shut down**

In [ ]:
#@title v-- Tap this if you play on Mobile { display-mode: "form" }
%%html
<b>Ensure this audio is playing to prevent colab from shutting down, then start KoboldCpp below</b><br/>
<audio autoplay="" src="https://raw.githubusercontent.com/KoboldAI/KoboldAI-Client/main/colab/silence.m4a" loop controls>

In [ ]:
#@title v-- Start the OpenRP Bridge (queued chat API with threaded workers) { display-mode: "form" }
#@markdown **Handles chat just like `openRP-API.py`, but on top of your KoboldCpp run:**<br>
#@markdown &nbsp;&nbsp;• **`POST /chat`** with `{"chat_completion": [{"role": "...", "content": "..."}, ...]}` -> queues the request and instantly answers its `request_id`<br>
#@markdown &nbsp;&nbsp;• **`POST /response`** with `{"request_id": "..."}` -> poll this: `processing`, then `completed` (with `response`) or `failed` (with `error`)<br><br>
#@markdown Instead of a blocking chat_completion handler, daemon worker threads drain the queue and forward the messages to the local KoboldCpp server, so the HTTP endpoints never wait on generation. Run this cell **before** the KoboldCpp cell below - the bridge keeps waiting until KoboldCpp on port 5001 is loaded.<br>
#@markdown If you were using the NanoGPT client from openRP-API.py: point it at the Public API URL and POST here, the request/response contract is identical.<br>
#@markdown <hr>
LaunchBridge = True #@param {type:"boolean"}
BridgePort = 5002 #@param {type:"integer"}
BridgeWorkers = 2 #@param {type:"integer"}
BridgePublicTunnel = True #@param {type:"boolean"}
BridgeGenTimeout = 3600 #@param {type:"integer"}

if LaunchBridge:
  import re, os, time, subprocess
  import requests

  BridgePort = int(BridgePort)
  BridgeWorkers = max(1, int(BridgeWorkers))

  print("Setting up the OpenRP bridge...")
  !pip -q install fastapi uvicorn > /dev/null 2>&1

  bridge_lines = [
    '#!/usr/bin/env python3',
    '"""openRP-API.py port for KoboldCpp - queued chat API with threaded workers.',
    '',
    'Endpoints (same contract as openRP-API.py):',
    '',
    '  POST /chat      body: {"chat_completion": [{"role": "...", "content": "..."}, ...]}',
    '                  -> {"status": "queued", "request_id": "...", "messages": N}',
    '                  errors -> {"status": "error", "message": "..."}',
    '',
    '  POST /response  body: {"request_id": "..."}',
    '                  -> {"status": "processing", "request_id": "..."}',
    '                  -> {"status": "completed", "request_id": "...", "response": "..."}',
    '                  -> {"status": "failed", "request_id": "...", "error": "..."}',
    '',
    'Nothing blocks on the way in: jobs are pushed onto a queue and daemon worker',
    'threads drain it and forward the messages to the local KoboldCpp server',
    '(its /v1/chat/completions endpoint), like the cloud OpenAI client in',
    'openRP-API.py, just pointed at a local model host. KoboldCpp keeps serving',
    'its usual UI/API; this queue lives on top of it.',
    '"""',
    '',
    'import os',
    'import time',
    'import uuid',
    'import json',
    'import threading',
    '',
    'import requests',
    'from fastapi import FastAPI, Request',
    '',
    'KCPP_HOST = os.getenv("KCPP_HOST", "http://127.0.0.1:5001").rstrip("/")',
    'GEN_TIMEOUT = int(os.getenv("BRIDGE_GEN_TIMEOUT", "3600"))',
    'KCPP_WAIT = int(os.getenv("BRIDGE_KCPP_WAIT", "10800"))',
    'WORKERS = int(os.getenv("BRIDGE_WORKERS", "2"))',
    '',
    '# Extra options a /chat request may carry besides the messages (optional).',
    'PASSTHROUGH_KEYS = ("temperature", "top_p", "top_k", "min_p", "tfs", "typical",',
    '                    "max_tokens", "stop", "seed", "presence_penalty",',
    '                    "frequency_penalty", "repetition_penalty")',
    '',
    'app = FastAPI()',
    '',
    'chat_queue = []',
    'chat_data = {}',
    'queue_lock = threading.Lock()',
    '',
    'def banner(title, *lines):',
    '    print("")',
    '    print("=" * 40)',
    '    print(title)',
    '    for line in lines:',
    '        print(line)',
    '    print("=" * 40)',
    '',
    '@app.get("/")',
    'async def index():',
    '    return {"status": "ok", "service": "openrp-bridge", "workers": WORKERS}',
    '',
    '@app.post("/chat")',
    'async def chat(req: Request):',
    '    # --------------------------------------------------------',
    '    # Receive body',
    '    # --------------------------------------------------------',
    '    raw = (await req.body()).decode("utf-8", "replace")',
    '',
    '    # --------------------------------------------------------',
    '    # Parse body',
    '    # --------------------------------------------------------',
    '    try:',
    '        body = json.loads(raw)',
    '    except json.JSONDecodeError as e:',
    '        return {"status": "error", "message": "Invalid JSON: %s" % e}',
    '',
    '    if not isinstance(body, dict):',
    '        return {"status": "error", "message": "Body must be a JSON object"}',
    '',
    '    # --------------------------------------------------------',
    '    # Get chat completion',
    '    # --------------------------------------------------------',
    '    chat_completion = body.get("chat_completion")',
    '',
    '    if chat_completion is None:',
    '        return {"status": "error", "message": "Missing chat_completion"}',
    '',
    '    # --------------------------------------------------------',
    '    # Make sure it is an array',
    '    # --------------------------------------------------------',
    '    if not isinstance(chat_completion, list):',
    '        return {"status": "error", "message": "chat_completion is not an array"}',
    '',
    '    if not chat_completion:',
    '        return {"status": "error", "message": "chat_completion is empty"}',
    '',
    '    # Any sampling options the request may carry ride along.',
    '    extras = {k: body[k] for k in PASSTHROUGH_KEYS if k in body}',
    '',
    '    # --------------------------------------------------------',
    '    # Create request ID',
    '    # --------------------------------------------------------',
    '    request_id = str(uuid.uuid4())',
    '',
    '    # --------------------------------------------------------',
    '    # Store request',
    '    # --------------------------------------------------------',
    '    chat_data[request_id] = {',
    '        "request_id": request_id,',
    '        "status": "queued",',
    '        "messages": chat_completion,',
    '        "extras": extras,',
    '        "response": None,',
    '        "error": None,',
    '    }',
    '',
    '    # --------------------------------------------------------',
    '    # Add to queue',
    '    # --------------------------------------------------------',
    '    with queue_lock:',
    '        chat_queue.append(request_id)',
    '',
    '    banner("New chat request",',
    '           "Request ID: %s" % request_id,',
    '           "Messages: %d" % len(chat_completion),',
    '           "Queue: %d" % len(chat_queue))',
    '',
    '    # --------------------------------------------------------',
    '    # Return immediately',
    '    # --------------------------------------------------------',
    '    return {"status": "queued", "request_id": request_id, "messages": len(chat_completion)}',
    '',
    '@app.post("/response")',
    'async def response(req: Request):',
    '    raw = (await req.body()).decode("utf-8", "replace")',
    '',
    '    try:',
    '        body = json.loads(raw)',
    '    except json.JSONDecodeError as e:',
    '        return {"status": "error", "message": "Invalid JSON: %s" % e}',
    '',
    '    if not isinstance(body, dict):',
    '        return {"status": "error", "message": "Body must be a JSON object"}',
    '',
    '    request_id = body.get("request_id")',
    '',
    '    if not request_id or request_id not in chat_data:',
    '        return {"status": "not_found"}',
    '',
    '    data = chat_data[request_id]',
    '',
    '    # --------------------------------------------------------',
    '    # Queued / processing',
    '    # --------------------------------------------------------',
    '    if data["status"] in ("queued", "processing"):',
    '        return {"status": "processing", "request_id": request_id}',
    '',
    '    # --------------------------------------------------------',
    '    # Completed',
    '    # --------------------------------------------------------',
    '    if data["status"] == "completed":',
    '        return {"status": "completed", "request_id": request_id, "response": data["response"]}',
    '',
    '    # --------------------------------------------------------',
    '    # Failed',
    '    # --------------------------------------------------------',
    '    return {"status": "failed", "request_id": request_id, "error": data["error"]}',
    '',
    'def wait_for_koboldcpp():',
    '    # Be patient: the KoboldCpp cell may still be downloading and loading the',
    '    # model when the first requests arrive.',
    '    polled = False',
    '    deadline = time.time() + KCPP_WAIT',
    '    while time.time() < deadline:',
    '        try:',
    '            r = requests.get(KCPP_HOST + "/api/extra/version", timeout=10)',
    '            if r.ok and r.json().get("llm"):',
    '                if polled:',
    '                    print("KoboldCpp is online, releasing the queue.")',
    '                return True',
    '        except Exception:',
    '            pass',
    '        polled = True',
    '        time.sleep(2)',
    '    return False',
    '',
    'def generate_chat(messages, extras):',
    '    if not wait_for_koboldcpp():',
    '        raise RuntimeError("KoboldCpp at %s never came online (waited %d s)" % (KCPP_HOST, KCPP_WAIT))',
    '',
    '    payload = {"model": "koboldcpp", "messages": messages}',
    '    payload.update(extras)',
    '',
    '    r = requests.post(KCPP_HOST + "/v1/chat/completions", json=payload, timeout=GEN_TIMEOUT)',
    '    r.raise_for_status()',
    '',
    '    try:',
    '        content = r.json()["choices"][0]["message"]["content"]',
    '    except (KeyError, IndexError, TypeError):',
    '        raise RuntimeError("KoboldCpp sent a malformed response: %s" % str(r.text)[:500])',
    '    return content',
    '',
    'def prune_chat_data():',
    '    # Drop the oldest finished jobs so long-running bridges stay slim,',
    '    # but never touch ones that are queued or still processing.',
    '    if len(chat_data) <= 200:',
    '        return',
    '    finished = [k for k, v in chat_data.items() if v["status"] in ("completed", "failed")]',
    '    for request_id in finished[:len(chat_data) - 200]:',
    '        chat_data.pop(request_id, None)',
    '',
    'def chat_worker():',
    '    print("Chat worker started")',
    '    while True:',
    '        with queue_lock:',
    '            request_id = chat_queue.pop(0) if chat_queue else None',
    '',
    '        if not request_id:',
    '            time.sleep(0.5)',
    '            continue',
    '',
    '        data = chat_data[request_id]',
    '',
    '        data["status"] = "processing"',
    '',
    '        banner("Processing chat request", "Request ID: %s" % request_id)',
    '',
    '        try:',
    '            start = time.time()',
    '            result = generate_chat(data["messages"], data["extras"])',
    '',
    '            data["response"] = result',
    '            data["status"] = "completed"',
    '',
    '            banner("Chat completed: %s" % request_id,',
    '                   "Took %.1f s, queue: %d" % (time.time() - start, len(chat_queue)),',
    '                   "Response:", result)',
    '',
    '            prune_chat_data()',
    '        except Exception as e:',
    '            banner("Chat failed: %s" % request_id, repr(e))',
    '',
    '            data["status"] = "failed"',
    '            data["error"] = str(e) or repr(e)',
    '',
    '            prune_chat_data()',
    '',
    'for i in range(WORKERS):',
    '    threading.Thread(target=chat_worker, daemon=True).start()',
    '',
    'if __name__ == "__main__":',
    '    import uvicorn',
    '    uvicorn.run(app, host="0.0.0.0", port=int(os.getenv("BRIDGE_PORT", "5002")))',
  ]

  bridge_path = "/content/openrp_bridge.py"
  with open(bridge_path, "w", encoding="utf-8") as f:
      f.write('\n'.join(bridge_lines) + '\n')

  !pkill -f openrp_bridge.py || true
  !sleep 1
  !BRIDGE_PORT=$BridgePort BRIDGE_WORKERS=$BridgeWorkers BRIDGE_GEN_TIMEOUT=$BridgeGenTimeout nohup python /content/openrp_bridge.py > /content/openrp_bridge.log 2>&1 &

  deadline = time.time() + 60
  while time.time() < deadline:
      try:
          if requests.get("http://127.0.0.1:%d/" % BridgePort, timeout=3).json().get("service") == "openrp-bridge":
              print("OpenRP bridge is listening on port %d." % BridgePort)
              break
      except Exception:
          time.sleep(1)
  else:
      print("Bridge gave no answer yet, check /content/openrp_bridge.log for errors:")
      !tail -n 40 /content/openrp_bridge.log

  bridge_url = None
  if BridgePublicTunnel:
    if not os.path.exists("/content/cloudflared"):
      !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
      !chmod +x /content/cloudflared

    already = subprocess.run(["pgrep", "-f", "cloudflared tunnel --url http://localhost:%d" % BridgePort], stdout=subprocess.PIPE, stderr=subprocess.PIPE).stdout.strip()
    if already:
      print("Existing cloudflared tunnel for port %d found, reusing it." % BridgePort)
    else:
      !nohup /content/cloudflared tunnel --url http://localhost:$BridgePort --no-autoupdate > /content/openrp_tunnel.log 2>&1 &
      time.sleep(2)

    bridge_url = None
    tunnel_log = ""
    for _attempt in range(10):
      try:
        with open("/content/openrp_tunnel.log", "r", errors="ignore") as fh:
          tunnel_log = fh.read()
      except OSError:
        tunnel_log = ""
      m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', tunnel_log)
      if m:
        bridge_url = m.group(0)
        break
      time.sleep(2)
    if not bridge_url:
      print("No trycloudflare URL in /content/openrp_tunnel.log yet, it may still be connecting:")
      !tail -n 20 /content/openrp_tunnel.log
  else:
    print("BridgePublicTunnel is off, the API stays on this Colab VM only.")

  local_url = "http://127.0.0.1:%d" % BridgePort
  api_url = bridge_url or local_url
  print()
  print("================================")
  print("OpenRP chat bridge")
  print("Local  API :", local_url)
  print("Public API :", bridge_url if bridge_url else "(tunnel disabled or starting)")
  print("Workers    :", BridgeWorkers)
  print()
  print("Queue a chat:  POST %s/chat" % api_url)
  print('  body: {"chat_completion": [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}]}')
  print('  reply: {"status": "queued", "request_id": "...", "messages": N}')
  print("Poll result:   POST %s/response" % api_url)
  print('  body: {"request_id": "<request_id>"}')
  print('  reply: processing | completed (with response) | failed (with error)')
  print()
  print("Job activity prints to /content/openrp_bridge.log")
  print("The bridge waits for KoboldCpp on port 5001, so start the cell below right away.")
  print("================================")

else:
  print("OpenRP bridge disabled (LaunchBridge = False).")

In [ ]:
# @title <b>v-- Enter your model below and then click this to start Koboldcpp</b> {"display-mode":"form"}
#@markdown You will need to restart KoboldCpp with the play button when you make changes to these settings.

#@markdown ### Select a quick preset template (optional)
Template = "Gemma4 E4B Uncensored (General)" # @param ["Gemma4 E4B Uncensored (General)", "Tiefighter 13B (General)","Estopia 13B (General)","Qwen3 VL 8B (Vision)","Cydonia 24B (Roleplay)","Broken Tutu 24B (Roleplay)","Dans PersonalityEngine 24B (Roleplay)","Harbinger 24B (Instruct)","Muse 12B (Instruct)","Mistral Small 24B (Corporate)","GPT OSS 20B (Corporate)","Gemma3 12B (Corporate)","GLM4 9b chat (Corporate)","Z-Image Turbo (Image)","Flux Klein 4B (Image)"]


#@markdown ### Or, choose manually by <u>selecting checkboxes and dropdown</u> below (Overwrites above templates!)

#@markdown <hr>
LoadTextModel = False #@param {type:"boolean"}
Model = "https://huggingface.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive/resolve/main/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q5_K_M.gguf" #@param ["https://huggingface.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive/resolve/main/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q5_K_M.gguf","https://huggingface.co/KoboldAI/LLaMA2-13B-Tiefighter-GGUF/resolve/main/LLaMA2-13B-Tiefighter.Q4_K_S.gguf","https://huggingface.co/KoboldAI/LLaMA2-13B-Estopia-GGUF/resolve/main/LLaMA2-13B-Estopia.Q4_K_S.gguf","https://huggingface.co/KoboldAI/Llama-3.1-8B-BookAdventures-GGUF/resolve/main/Llama-3.1-8B-BookAdventures.Q6_K.gguf","https://huggingface.co/bartowski/TheDrummer_Cydonia-24B-v4.2.0-GGUF/resolve/main/TheDrummer_Cydonia-24B-v4.2.0-Q4_K_S.gguf","https://huggingface.co/mradermacher/Broken-Tutu-24B-GGUF/resolve/main/Broken-Tutu-24B.Q4_K_S.gguf","https://huggingface.co/bartowski/PocketDoc_Dans-PersonalityEngine-V1.3.0-24b-GGUF/resolve/main/PocketDoc_Dans-PersonalityEngine-V1.3.0-24b-Q4_K_S.gguf","https://huggingface.co/LatitudeGames/Harbinger-24B-GGUF/resolve/main/Harbinger-24B-Q4_K_S.gguf","https://huggingface.co/LatitudeGames/Muse-12B-GGUF/resolve/main/Muse-12B-Q4_K_S.gguf","https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-Q6_K.gguf","https://huggingface.co/unsloth/Mistral-Small-3.2-24B-Instruct-2506-GGUF/resolve/main/Mistral-Small-3.2-24B-Instruct-2506-Q4_K_S.gguf","https://huggingface.co/ggml-org/gpt-oss-20b-GGUF/resolve/main/gpt-oss-20b-mxfp4.gguf","https://huggingface.co/KoboldAI/Llama-3.1-8B-BookAdventures-GGUF/resolve/main/Llama-3.1-8B-BookAdventures.Q6_K.gguf","https://huggingface.co/bartowski/google_gemma-3-12b-it-GGUF/resolve/main/google_gemma-3-12b-it-Q4_K_S.gguf","https://huggingface.co/unsloth/gemma-3n-E4B-it-GGUF/resolve/main/gemma-3n-E4B-it-Q6_K.gguf","https://huggingface.co/unsloth/GLM-4-9B-0414-GGUF/resolve/main/GLM-4-9B-0414-Q6_K.gguf","https://huggingface.co/mradermacher/Fimbulvetr-11B-v2-GGUF/resolve/main/Fimbulvetr-11B-v2.Q4_K_S.gguf","https://huggingface.co/TheBloke/MythoMax-L2-13B-GGUF/resolve/main/mythomax-l2-13b.Q4_K_M.gguf","https://huggingface.co/TheBloke/ReMM-SLERP-L2-13B-GGUF/resolve/main/remm-slerp-l2-13b.Q4_K_M.gguf","https://huggingface.co/TheBloke/Xwin-LM-13B-v0.2-GGUF/resolve/main/xwin-lm-13b-v0.2.Q4_K_M.gguf","https://huggingface.co/mradermacher/mini-magnum-12b-v1.1-GGUF/resolve/main/mini-magnum-12b-v1.1.Q4_K_S.gguf","https://huggingface.co/TheBloke/Stheno-L2-13B-GGUF/resolve/main/stheno-l2-13b.Q4_K_M.gguf","https://huggingface.co/TheBloke/MythoMax-L2-Kimiko-v2-13B-GGUF/resolve/main/mythomax-l2-kimiko-v2-13b.Q4_K_M.gguf","https://huggingface.co/bartowski/Rocinante-12B-v1.1-GGUF/resolve/main/Rocinante-12B-v1.1-Q4_K_S.gguf","https://huggingface.co/TheBloke/MistRP-Airoboros-7B-GGUF/resolve/main/mistrp-airoboros-7b.Q4_K_S.gguf","https://huggingface.co/TheBloke/airoboros-mistral2.2-7B-GGUF/resolve/main/airoboros-mistral2.2-7b.Q4_K_S.gguf","https://huggingface.co/concedo/KobbleTinyV2-1.1B-GGUF/resolve/main/KobbleTiny-Q4_K.gguf","https://huggingface.co/grimjim/kukulemon-7B-GGUF/resolve/main/kukulemon-7B.Q8_0.gguf","https://huggingface.co/mradermacher/LemonKunoichiWizardV3-GGUF/resolve/main/LemonKunoichiWizardV3.Q4_K_M.gguf","https://huggingface.co/Lewdiculous/Kunoichi-DPO-v2-7B-GGUF-Imatrix/resolve/main/Kunoichi-DPO-v2-7B-Q4_K_M-imatrix.gguf","https://huggingface.co/mradermacher/L3-8B-Stheno-v3.2-i1-GGUF/resolve/main/L3-8B-Stheno-v3.2.i1-Q4_K_M.gguf","https://huggingface.co/Lewdiculous/Llama-3-Lumimaid-8B-v0.1-OAS-GGUF-IQ-Imatrix/resolve/main/v2-Llama-3-Lumimaid-8B-v0.1-OAS-Q4_K_M-imat.gguf","https://huggingface.co/bartowski/NeuralDaredevil-8B-abliterated-GGUF/resolve/main/NeuralDaredevil-8B-abliterated-Q4_K_M.gguf","https://huggingface.co/bartowski/L3-8B-Lunaris-v1-GGUF/resolve/main/L3-8B-Lunaris-v1-Q4_K_M.gguf","https://huggingface.co/mradermacher/L3-Umbral-Mind-RP-v2.0-8B-GGUF/resolve/main/L3-Umbral-Mind-RP-v2.0-8B.Q4_K_M.gguf","https://huggingface.co/bartowski/TheDrummer_Cydonia-24B-v2-GGUF/resolve/main/TheDrummer_Cydonia-24B-v2-Q4_K_S.gguf","https://huggingface.co/bartowski/PocketDoc_Dans-PersonalityEngine-V1.2.0-24b-GGUF/resolve/main/PocketDoc_Dans-PersonalityEngine-V1.2.0-24b-IQ4_XS.gguf","https://huggingface.co/mradermacher/Tlacuilo-12B-GGUF/resolve/main/Tlacuilo-12B.Q4_K_S.gguf"] {"allow-input":true}
MdCommand = "" #@markdown <br>
Layers = "Auto" #@param ["Auto","999"]{allow-input: true}
ContextSize = "4096" #@param ["4096","8192","12288","16384","24576","32768","40960"] {allow-input: true}

#@markdown <hr>
LoadVisionMMProjector = False #@param {type:"boolean"}
Mmproj = "https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/resolve/main/mmproj-BF16.gguf" #@param ["https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/resolve/main/mmproj-BF16.gguf","https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/mmproj-BF16.gguf","https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-13b-mmproj-v1.5.Q4_1.gguf","https://huggingface.co/koboldcpp/mmproj/resolve/main/mistral-7b-mmproj-v1.5-Q4_1.gguf","https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-7b-mmproj-v1.5-Q4_0.gguf","https://huggingface.co/koboldcpp/mmproj/resolve/main/LLaMA3-8B_mmproj-Q4_1.gguf"]{allow-input: true}
VCommand = ""

#@markdown <hr>
LoadImgModel = False #@param {type:"boolean"}
ImgModel = "https://huggingface.co/koboldcpp/imgmodel/resolve/main/picx_real_q4_0.gguf" #@param ["https://huggingface.co/koboldcpp/imgmodel/resolve/main/picx_real_q4_0.gguf","https://huggingface.co/koboldcpp/imgmodel/resolve/main/imgmodel_ftuned_q4_0.gguf","https://huggingface.co/koboldcpp/imgmodel/resolve/main/dreamshaperXL_v2_q4_k.gguf"]{allow-input: true}
#@markdown <br>
ImgClip1 = "" #@param [""]{allow-input: true}
ImgVAE = "" #@param [""]{allow-input: true}
SCommand = ""

#@markdown <hr>
LoadSpeechModel = False #@param {type:"boolean"}
SpeechModel = "https://huggingface.co/koboldcpp/whisper/resolve/main/whisper-base.en-q5_1.bin" #@param ["https://huggingface.co/koboldcpp/whisper/resolve/main/whisper-base.en-q5_1.bin"]{allow-input: true}
WCommand = ""

#@markdown <hr>
LoadTTSModel = False #@param {type:"boolean"}
TTSModel = "https://huggingface.co/koboldcpp/tts/resolve/main/Kokoro_no_espeak_Q4.gguf" #@param ["https://huggingface.co/koboldcpp/tts/resolve/main/OuteTTS-0.2-500M-Q4_0.gguf","https://huggingface.co/koboldcpp/tts/resolve/main/Kokoro_no_espeak_Q4.gguf"]{allow-input: true}
WavTokModel = "" #@param ["https://huggingface.co/koboldcpp/tts/resolve/main/WavTokenizer-Large-75-Q4_0.gguf"]{allow-input: true}
TTSCommand = ""

#@markdown <hr>
LoadEmbeddingsModel = False #@param {type:"boolean"}
EmbeddingsModel = "https://huggingface.co/yixuan-chia/snowflake-arctic-embed-s-GGUF/resolve/main/snowflake-arctic-embed-s-Q4_0.gguf" #@param ["https://huggingface.co/yixuan-chia/snowflake-arctic-embed-s-GGUF/resolve/main/snowflake-arctic-embed-s-Q4_0.gguf"]{allow-input: true}
ECommand = ""

#@markdown <hr>
#@markdown Currently only AceStep 1.5 is supported, will use it if picked
LoadMusicModel = False #@param {type:"boolean"}
MusicModel = "--musiclowvram --musicllm https://huggingface.co/koboldcpp/music/resolve/main/acestep-5Hz-lm-1.7B-BF16.gguf --musicembeddings https://huggingface.co/koboldcpp/music/resolve/main/Ace-Qwen3-Embedding-0.6B-BF16.gguf --musicdiffusion https://huggingface.co/koboldcpp/music/resolve/main/acestep-v15-sftturbo50-Q8_0.gguf --musicvae https://huggingface.co/koboldcpp/music/resolve/main/ace-vae-BF16.gguf"
MuCommand = ""

#@markdown <hr>
#@markdown Deletes all models, forcing a fresh redownload when you restart colab.
DeleteExistingModels = True #@param {type:"boolean"}

#@markdown <hr>
#@markdown Enables multiplayer sessions, simply load the link and click join multiplayer.
Multiplayer = False #@param {type:"boolean"}
MPCommand = ""

#@markdown <hr>
#@markdown This enables saving stories directly to your google drive. You will have to grant permissions, and then you can access the saves from the "KoboldCpp Server Storage" option.
AllowSaveToGoogleDrive = False #@param {type:"boolean"}
SavGdriveCommand = ""

#@markdown <hr>
#@markdown Only select the following box if regular cloudflare tunnel fails to work. It will generate an inferior localtunnel tunnel, which you can use after entering a password.
MakeLocalTunnelFallback = False #@param {type:"boolean"}

CustomCtxSize = ""
CustomMmproj = ""
Customized = False
has_checked_box = any((LoadTextModel, LoadVisionMMProjector, LoadImgModel, LoadSpeechModel,LoadTTSModel, LoadEmbeddingsModel))
if not has_checked_box:
  Model = ""
  ImgModel = ""
  ImgClip1 = ""
  ImgVAE = ""
  Mmproj = ""
  if Template == "Gemma4 E4B Uncensored (General)":
    Customized = True
    Model = "https://huggingface.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive/resolve/main/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q5_K_M.gguf"
    CustomCtxSize = "40960"
    CustomMmproj = "https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/resolve/main/mmproj-BF16.gguf"
  if Template == "Tiefighter 13B (General)":
    Customized = True
    Model = "https://huggingface.co/KoboldAI/LLaMA2-13B-Tiefighter-GGUF/resolve/main/LLaMA2-13B-Tiefighter.Q4_K_S.gguf"
    CustomCtxSize = "4096"
    CustomMmproj = "https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-13b-mmproj-v1.5.Q4_1.gguf"
  if Template == "Estopia 13B (General)":
    Customized = True
    Model = "https://huggingface.co/KoboldAI/LLaMA2-13B-Estopia-GGUF/resolve/main/LLaMA2-13B-Estopia.Q4_K_S.gguf"
    CustomCtxSize = "4096"
    CustomMmproj = "https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-13b-mmproj-v1.5.Q4_1.gguf"
  if Template == "Cydonia 24B (Roleplay)":
    Customized = True
    Model = "https://huggingface.co/bartowski/TheDrummer_Cydonia-24B-v4.2.0-GGUF/resolve/main/TheDrummer_Cydonia-24B-v4.2.0-Q4_K_S.gguf"
    CustomCtxSize = "10240"
  if Template == "Broken Tutu 24B (Roleplay)":
    Customized = True
    Model = "https://huggingface.co/mradermacher/Broken-Tutu-24B-GGUF/resolve/main/Broken-Tutu-24B.Q4_K_S.gguf"
    CustomCtxSize = "10240"
  if Template == "Dans PersonalityEngine 24B (Roleplay)":
    Customized = True
    Model = "https://huggingface.co/bartowski/PocketDoc_Dans-PersonalityEngine-V1.2.0-24b-GGUF/resolve/main/PocketDoc_Dans-PersonalityEngine-V1.2.0-24b-IQ4_XS.gguf"
    CustomCtxSize = "10240"
  if Template == "Harbinger 24B (Instruct)":
    Customized = True
    Model = "https://huggingface.co/LatitudeGames/Harbinger-24B-GGUF/resolve/main/Harbinger-24B-Q4_K_S.gguf"
    CustomCtxSize = "10240"
  if Template == "Muse 12B (Instruct)":
    Customized = True
    Model = "https://huggingface.co/LatitudeGames/Muse-12B-GGUF/resolve/main/Muse-12B-Q4_K_S.gguf"
    CustomCtxSize = "49152"
  if Template == "Qwen3 VL 8B (Vision)":
    Customized = True
    Model = "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-Q6_K.gguf"
    CustomMmproj = "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/mmproj-BF16.gguf"
    CustomCtxSize = "32768"
  if Template == "Mistral Small 24B (Corporate)":
    Customized = True
    Model = "https://huggingface.co/unsloth/Mistral-Small-3.2-24B-Instruct-2506-GGUF/resolve/main/Mistral-Small-3.2-24B-Instruct-2506-Q4_K_S.gguf"
    CustomCtxSize = "10240"
  if Template == "GPT OSS 20B (Corporate)":
    Customized = True
    Model = "https://huggingface.co/ggml-org/gpt-oss-20b-GGUF/resolve/main/gpt-oss-20b-mxfp4.gguf"
    CustomCtxSize = "32768"
  if Template == "Gemma3 12B (Corporate)":
    Customized = True
    Model = "https://huggingface.co/bartowski/google_gemma-3-12b-it-GGUF/resolve/main/google_gemma-3-12b-it-Q4_K_S.gguf"
    CustomCtxSize = "16384"
  if Template == "GLM4 9b chat (Corporate)":
    Customized = True
    Model = "https://huggingface.co/unsloth/GLM-4-9B-0414-GGUF/resolve/main/GLM-4-9B-0414-Q6_K.gguf"
    CustomCtxSize = "49152"
  if Template == "Z-Image Turbo (Image)":
    ImgModel = "https://huggingface.co/leejet/Z-Image-Turbo-GGUF/resolve/main/z_image_turbo-Q4_0.gguf"
    ImgClip1 = "https://huggingface.co/unsloth/Qwen3-4B-Instruct-2507-GGUF/resolve/main/Qwen3-4B-Instruct-2507-Q4_K_S.gguf"
    ImgVAE = "https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/ae.safetensors"
    Customized = True
  if Template == "Flux Klein 4B (Image)":
    ImgModel = "https://huggingface.co/leejet/FLUX.2-klein-4B-GGUF/resolve/main/flux-2-klein-4b-Q4_0.gguf"
    ImgClip1 = "https://huggingface.co/unsloth/Qwen3-4B-GGUF/resolve/main/Qwen3-4B-Q4_K_M.gguf"
    ImgVAE = "https://huggingface.co/koboldcpp/GGUFDumps/resolve/main/flux2vae.safetensors"
    Customized = True

if not Customized or Template=="" or has_checked_box:
  print("== Manual CHECKBOX OVERRIDE was detected, using manual selections ==")
  print(f"LoadTextModel: {LoadTextModel}")
  print(f"LoadVisionMMProjector: {LoadVisionMMProjector}")
  print(f"LoadImgModel: {LoadImgModel}")
  print(f"LoadSpeechModel: {LoadSpeechModel}")
  print(f"LoadTTSModel: {LoadTTSModel}")
  print(f"LoadEmbeddingsModel: {LoadEmbeddingsModel}")
elif Customized:
  print("== No checkbox override selected, loading a quick preset template! ==")
  print(f"CHOSEN PRSET: {Template}")
  print("If you want to load your own model selection, MAKE SURE you check one of the loader checkboxes...")
  if Model:
    LoadTextModel = True
    Layers = 999
  if ImgModel:
    LoadImgModel = True
  if CustomMmproj!="":
    Mmproj = CustomMmproj
    LoadVisionMMProjector = True
  if CustomCtxSize!="":
    ContextSize = CustomCtxSize

if Layers=="Auto":
  Layers = -1

print("=== STARTING THE LOAD ===")
if LoadTextModel:
  print(f"[Text Model] {Model}")
if LoadVisionMMProjector:
  print(f"[Vision MMProj] {Mmproj}")
if LoadImgModel:
  print(f"[Image Model] {ImgModel}")
  if ImgClip1:
    print(f"  └─ Clip: {ImgClip1}")
  if ImgVAE:
    print(f"  └─ VAE: {ImgVAE}")
if LoadSpeechModel:
  print(f"[Speech Model] {SpeechModel}")
if LoadTTSModel:
  print(f"[TTS Model] {TTSModel}")
  if WavTokModel:
    print(f"  └─ WavTokenizer: {WavTokModel}")
if LoadEmbeddingsModel:
  print(f"[Embeddings Model] {EmbeddingsModel}")

import os, glob
if not os.path.isfile("/opt/bin/nvidia-smi"):
  raise RuntimeError("⚠️Colab did not give you a GPU due to usage limits, this can take a few hours before they let you back in. Check out https://lite.koboldai.net for a free alternative (that does not provide an API link but can load KoboldAI saves and chat cards) or subscribe to Colab Pro for immediate access.⚠️")

if AllowSaveToGoogleDrive:
  print("Attempting to request access to save to your google drive...")
  try:
    from google.colab import drive
    import os, json
    drive.mount('/content/drive', force_remount=True)
    if not os.path.exists("/content/drive/MyDrive"):
      raise RuntimeError("Google Drive mount failed. Please grant permissions and try again.")
    kcppdir = '/content/drive/MyDrive/koboldcpp_data'
    os.makedirs(kcppdir, exist_ok=True)
    savedatapath = os.path.join(kcppdir, "koboldcpp_save_db.jsondb")
    if not os.path.exists(savedatapath):
        settings_data = {}
        with open(savedatapath, "w") as json_file:
            json.dump(settings_data, json_file, indent=4)
        print(f"Created new koboldcpp_save_db.jsondb at {savedatapath}")
    else:
        print(f"Loading saved data at {savedatapath}")
    SavGdriveCommand = f" --savedatafile {savedatapath}"
  except Exception as e:
    print(f"⚠️ Error: {e}")
    print("Please ensure you grant Google Drive permissions and try again.")

%cd /content
if Model and LoadTextModel:
  MdCommand = f"--model {Model}"
else:
  MdCommand = ""
if Mmproj and LoadVisionMMProjector:
  VCommand = f"--mmproj {Mmproj}"
else:
  VCommand = ""
if ImgModel and LoadImgModel:
  SCommand = f"--sdmodel {ImgModel} --sdthreads 4 --sdquant --sdclamped 768 --sdclipgpu"
  if ImgClip1 and ImgClip1!="":
    SCommand += f" --sdclip1 {ImgClip1}"
  if ImgVAE and ImgVAE!="":
    SCommand += f" --sdvae {ImgVAE}"
else:
  SCommand = ""
if SpeechModel and LoadSpeechModel:
  WCommand = f"--whispermodel {SpeechModel}"
else:
  WCommand = ""
if TTSModel and WavTokModel and LoadTTSModel:
  TTSCommand = f"--ttsmodel {TTSModel} --ttswavtokenizer {WavTokModel} --ttsgpu"
elif TTSModel and LoadTTSModel:
  TTSCommand = f"--ttsmodel {TTSModel} --ttsgpu"
else:
  TTSCommand = ""
if EmbeddingsModel and LoadEmbeddingsModel:
  ECommand = f"--embeddingsmodel {EmbeddingsModel}"
else:
  ECommand = ""
if LoadMusicModel:
  MuCommand = f"{MusicModel}"
else:
  MuCommand = ""
if Multiplayer:
  MPCommand = "--multiplayer"
else:
  MPCommand = ""

if DeleteExistingModels:
  print("Deleting all cached models to redownload...")
  patterns = ['*.gguf', '*.bin', '*.ggml']
  for pattern in patterns:
      for file_path in glob.glob(pattern):
          try:
              os.remove(file_path)
              print(f"Deleted: {file_path}")
          except Exception as e:
              print(f"Failed to delete {file_path}: {e}")


!echo Downloading KoboldCpp, please wait...
!wget -O dlfile.tmp https://github.com/LostRuins/koboldcpp/releases/latest/download/koboldcpp-linux-x64 && mv dlfile.tmp koboldcpp_linux || (echo "Primary failed, trying mirror..." && wget -O dlfile.tmp https://koboldai.org/cpplinuxcu12 && mv dlfile.tmp koboldcpp_linux)
!test -f koboldcpp_linux && echo Download Successful || echo Download Failed
!chmod +x ./koboldcpp_linux
!apt update
!apt install aria2 -y

if MakeLocalTunnelFallback:
  import urllib
  print("Trying to use LocalTunnel as a fallback tunnel (not so good)...")
  ltpw = urllib.request.urlopen('https://loca.lt/mytunnelpassword').read().decode('utf8').strip("\n")
  !nohup npx --yes localtunnel --port 5001 > lt.log 2>&1 &
  !sleep 8
  print("=================")
  print("(LocalTunnel Results)")
  !cat lt.log
  print(f"Please open the above link, and input the password '{ltpw}'\nYour KoboldCpp will start shortly...")
  print("=================")
  !sleep 10
!./koboldcpp_linux $MdCommand --usecuda 0 mmq --chatcompletionsadapter AutoGuess --multiuser 10 --flashattention --gpulayers $Layers --contextsize $ContextSize --websearch --quiet --remotetunnel $MPCommand $VCommand $SCommand $WCommand $TTSCommand $ECommand $MuCommand $SavGdriveCommand
